## 17_join_playground  (OLD workspace = Databricks Free Edition)

Hands-on join practice over the **10 relational base tables** created by
`00_setup_sample_data` — customer-360 star schema: **accounts / transactions /
tickets / campaigns**. Old workspace = 2-part `schema.table` names only (no
catalog); all tables are partitioned by `ingest_date`.

**Join ladder (2 -> 8 tables):**

| Cells | Tables | Join type | What it shows |
|-------|--------|-----------|---------------|
| c2 | customers + addresses, customers + accounts | LEFT / INNER | LEFT keeps unmatched customers; INNER drops them |
| c3 | customers -> accounts -> transactions | INNER (3) | aggregation through a bridge (spend per customer) |
| c4 | customers + addresses + accounts + subscriptions + products | INNER (5) | star-schema fan-out + dedup lesson |
| c5 | + support_tickets + agents + campaign_responses + campaigns | INNER (8) + RIGHT variant | 8-table chain; RIGHT JOIN keeps customers WITHOUT tickets |
| c6 | future-model hook | LEFT | transaction aggregates as extra training features |

PREREQ: run `00_setup_sample_data` first. Keep `PARTITION=all` to read every partition.

In [ ]:

%run ./lib/config_lib

dbutils.widgets.text("SCHEMA", "practice", "Schema with the practice base tables")
dbutils.widgets.text("PARTITION", "all", "all | latest | prev | YYYY-MM-DD")

CFG = setup_config(
    config_path="",
    run_date=dbutils.widgets.get("PARTITION").strip(),
    schema=dbutils.widgets.get("SCHEMA").strip(),
)

def T(simple):
    """2-part table name via CFG — the old workspace has no catalog."""
    return CFG.table(simple)

base_tables = ["practice_customers", "practice_addresses", "practice_accounts",
               "practice_products", "practice_subscriptions", "practice_transactions",
               "practice_support_tickets", "practice_agents", "practice_campaigns",
               "practice_campaign_responses"]
for t in base_tables:
    print(f"{T(t):50s} rows={spark.table(T(t)).count()}")


In [ ]:

# ---- 2-TABLE JOINS ---------------------------------------------------------
cust = T("practice_customers")
addr = T("practice_addresses")
acc  = T("practice_accounts")

# LEFT JOIN: every customer survives; customers can have home AND billing rows.
left_df = spark.sql(f"""
    SELECT c.customer_id, c.segment, a.address_type, a.city
    FROM {cust} c
    LEFT JOIN {addr} a ON c.customer_id = a.customer_id
    ORDER BY c.customer_id, a.address_type
""")
n_left = left_df.count()

# INNER JOIN: only rows with a match on BOTH sides.
inner_df = spark.sql(f"""
    SELECT c.customer_id, c.segment, a.account_id, a.account_type, a.balance
    FROM {cust} c
    INNER JOIN {acc} a ON c.customer_id = a.customer_id
    ORDER BY c.customer_id, a.account_id
""")
n_inner = inner_df.count()

n_cust = spark.table(cust).count()
print(f"customers = {n_cust}")
print(f"LEFT  customers x addresses : {n_left} rows  (> {n_cust} because 1 customer -> 1..2 addresses)")
print(f"INNER customers x accounts  : {n_inner} rows  (> {n_cust} because 1 customer -> 1..2 accounts)")
print()
print("LEFT keeps unmatched parents; INNER drops them. Both join keys resolve")
print("for every customer here (FK check in 00), so LEFT == INNER count would")
print("only hold for 1:1 relationships.")
display(inner_df.limit(10))


In [ ]:

# ---- 3-TABLE INNER JOIN (aggregation through a bridge) ---------------------
txns = T("practice_transactions")

spend_df = spark.sql(f"""
    SELECT c.customer_id, c.segment,
           COUNT(t.txn_id)              AS txn_cnt,
           ROUND(SUM(t.amount), 2)      AS total_spend
    FROM {cust} c
    INNER JOIN {acc} a  ON c.customer_id = a.customer_id
    INNER JOIN {txns} t ON a.account_id  = t.account_id
    GROUP BY c.customer_id, c.segment
    ORDER BY total_spend DESC
""")
print(f"customers -> accounts -> transactions (3-table INNER): {spend_df.count()} customers with spend")
print("The bridge table (accounts) carries the join from person to money.")
display(spend_df.limit(10))


In [ ]:

# ---- 5-TABLE STAR-SCHEMA JOIN (fan-out + dedup lesson) ---------------------
prods = T("practice_products")
subs  = T("practice_subscriptions")

five_df = spark.sql(f"""
    WITH subs_dedup AS (            -- bridge fan-out: dedup account x product pairs
        SELECT account_id, product_id
        FROM {subs}
        GROUP BY account_id, product_id
    )
    SELECT c.segment, p.category,
           COUNT(DISTINCT c.customer_id) AS customers,
           ROUND(SUM(t.amount), 2)       AS revenue
    FROM {cust} c
    INNER JOIN {acc}          a  ON c.customer_id = a.customer_id
    INNER JOIN subs_dedup     s  ON a.account_id  = s.account_id
    INNER JOIN {prods}        p  ON s.product_id  = p.product_id
    INNER JOIN {txns}         t  ON a.account_id  = t.account_id
                                AND t.product_id = s.product_id
    GROUP BY c.segment, p.category
    ORDER BY revenue DESC
""")
print("5 tables: customers + accounts + subscriptions + products + transactions")
print("Lesson: join a bridge table twice (or fan it out) and counts explode —")
print("dedup the bridge (subs_dedup) before aggregating money.")
display(five_df)


In [ ]:

# ---- 8-TABLE CHAIN + RIGHT JOIN variant ------------------------------------
tickets = T("practice_support_tickets")
agents  = T("practice_agents")
resps   = T("practice_campaign_responses")
camps   = T("practice_campaigns")

eight_df = spark.sql(f"""
    SELECT c.customer_id, ad.city, a.account_id, t.txn_id, t.amount,
           k.ticket_id, k.status, g.region, r.responded, cp.channel
    FROM {cust} c
    INNER JOIN {addr}   ad ON c.customer_id  = ad.customer_id
    INNER JOIN {acc}    a  ON c.customer_id  = a.customer_id
    INNER JOIN {txns}   t  ON a.account_id   = t.account_id
    LEFT  JOIN {tickets} k ON c.customer_id  = k.customer_id
    LEFT  JOIN {agents}  g ON k.agent_id     = g.agent_id
    LEFT  JOIN {resps}   r ON c.customer_id  = r.customer_id
    LEFT  JOIN {camps}   cp ON r.campaign_id = cp.campaign_id
    LIMIT 20
""")
print("8 tables in one statement: customers + addresses + accounts + transactions")
print("+ support_tickets + agents + campaign_responses + campaigns (display limited to 20 rows).")
n_eight = eight_df.count()
print(f"returned rows (limited): {n_eight}")

# RIGHT JOIN variant — the side that matters is the RIGHT table (customers):
# customers WITHOUT tickets (20 of them) survive a RIGHT JOIN but vanish in INNER.
right_df = spark.sql(f"""
    SELECT k.ticket_id, k.status, c.customer_id, c.segment
    FROM {tickets} k
    RIGHT JOIN {cust} c ON k.customer_id = c.customer_id
""")
n_right = right_df.count()
n_tk    = spark.table(tickets).count()
n_tk_cust = spark.sql(f"SELECT COUNT(DISTINCT customer_id) AS n FROM {tickets}").collect()[0]["n"]
print()
print(f"RIGHT tickets -> customers : {n_right} rows = {n_tk} ticket rows + "
      f"{n_right - n_tk} customers with NO tickets")
print(f"(customers having tickets: {n_tk_cust}/100 — the other {100 - n_tk_cust} only exist via RIGHT)")
display(right_df.orderBy("customer_id").limit(10))


In [ ]:

# ---- FUTURE-MODEL HOOK: relational aggregates as training features ---------
# The trainers read ONE table (practice_customers). To add behavior features
# later, aggregate the fact table per customer and LEFT JOIN it onto the
# training rows — same shape the jobs' task chain produces:
tx_feats = spark.sql(f"""
    SELECT c.customer_id,
           COUNT(t.txn_id)                        AS txn_count,
           ROUND(COALESCE(SUM(t.amount), 0), 2)   AS txn_total
    FROM {cust} c
    LEFT JOIN {acc} a  ON c.customer_id = a.customer_id
    LEFT JOIN {txns} t ON a.account_id  = t.account_id
    GROUP BY c.customer_id
""")

training_rows = CFG.partition_filter(
    spark.table(cust), table=cust)          # run_date=PARTITION honoured
enriched = training_rows.join(tx_feats, "customer_id", "left")

print(f"training frame + relational features: {enriched.count()} rows, "
      f"columns = {len(enriched.columns)}")
display(enriched.orderBy("customer_id").limit(10))
print()
print("NEXT (future): append txn_count / txn_total to FEATURES in lib/config_lib.py")
print("and re-run the trainers — jobs, export (30/35) and the lift (47) pick it up")
print("with zero further code changes.")


In [ ]:

print("=== JOIN PLAYGROUND SUMMARY ===")
print("- 2-table LEFT vs INNER: LEFT keeps unmatched parents, INNER drops them.")
print("- 3-table INNER through a bridge (accounts): person -> money aggregation.")
print("- 5-table star schema: dedup the bridge (fan-out explodes counts).")
print("- 8-table chain: LEFT for optional branches; RIGHT variant keeps")
print("  customers WITHOUT tickets that INNER would drop.")
print("- All names are 2-part schema.table (old workspace, no catalog);")
print("  on the NEW platform the same statements run with 3-part names")
print("  (catalog.schema.table) via CFG — see bundle/src/lib/config_lib.py.")
